# Run Stage 1 training

This notebook runs the corrected Stage 1 training script. By default it prints the resolved PartImageNet layout and runs a one-batch smoke/audit pass. Set `RUN_FULL_STAGE1_TRAIN = True` to launch the 18-epoch training command.

In [ ]:
import os, subprocess, shlex, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PARTIMAGENET_ROOT = os.environ.get('PARTIMAGENET_ROOT', '../full_hyco/PartImageNet')
CONFIG = 'configs/default.yaml'
SAVE_DIR = 'runs/stage1_default'
DEVICE = os.environ.get('STAGE1_DEVICE', 'auto')

RUN_PRINT_LAYOUT = True
RUN_SMOKE_ONLY = True
RUN_FULL_STAGE1_TRAIN = False

print('Project root:', PROJECT_ROOT)
print('PartImageNet root:', PARTIMAGENET_ROOT)

In [ ]:
def run(cmd):
    print('\nRunning:', ' '.join(shlex.quote(str(x)) for x in cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

base = [
    sys.executable, 'scripts/train_stage1.py',
    '--config', CONFIG,
    '--device', DEVICE,
    '--partimagenet-root', PARTIMAGENET_ROOT,
]

if RUN_PRINT_LAYOUT:
    run(base + ['--print-data-layout'])

if RUN_SMOKE_ONLY:
    run(base + [
        '--smoke-only',
        '--num-workers', '0',
        '--batch-size', '2',
        '--torch-threads', '1',
    ])

if RUN_FULL_STAGE1_TRAIN:
    run(base + [
        '--epochs', '18',
        '--save-dir', SAVE_DIR,
    ])

## Full training command

The direct terminal command is:

```bash
python scripts/train_stage1.py \
  --config configs/default.yaml \
  --device auto \
  --epochs 18 \
  --save-dir runs/stage1_default \
  --partimagenet-root ../full_hyco/PartImageNet
```

For a temporary workaround on an unpatched checkout, add `--no-amp`; with this patch, AMP training should run without the probability-domain BCE error.